# Giai đoạn 3: Mô phỏng Monte Carlo Mô hình Heston (Euler-Maruyama)

Notebook này nạp bộ tham số Heston tối ưu đã khớp ở Giai đoạn 2, thực hiện mô phỏng Monte Carlo (Euler-Maruyama Full Truncation) cho toàn bộ các hợp đồng quyền chọn, và đối chứng giá mô phỏng ngẫu nhiên với giá giải tích Fourier.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thêm thư mục cha vào sys.path để import các hàm từ src
sys.path.append(os.path.abspath('..'))

from src.math_utils import heston_analytical_price, heston_monte_carlo_price

# Cố định hạt giống ngẫu nhiên để đảm bảo tính tái lập (Reproducibility)
np.random.seed(42)

In [ ]:
# 1. Nạp tham số Heston lý tưởng đã calibrated từ Giai đoạn 2
with open("../data/processed/heston_parameters.json", "r") as f:
    heston_params = json.load(f)

v0 = heston_params["v0"]
kappa = heston_params["kappa"]
theta = heston_params["theta"]
xi = heston_params["xi"]
rho = heston_params["rho"]

print("🎉 Nạp thành công bộ tham số Heston:")
print(f"v0: {v0:.5f}, kappa: {kappa:.5f}, theta: {theta:.5f}, xi: {xi:.5f}, rho: {rho:.5f}")

In [ ]:
# 2. Nạp dữ liệu quyền chọn sạch
df_clean = pd.read_csv("../data/processed/02_btc_options_with_iv.csv")
print(f"📊 Đã nạp {len(df_clean)} hợp đồng để chạy kiểm thử Monte Carlo.")

In [ ]:
# 3. Tiến hành định giá bằng Monte Carlo (Chạy 20,000 đường đi cho mỗi hợp đồng)
print("🔄 Đang chạy mô phỏng Monte Carlo ngẫu nhiên (Quá trình này có thể mất một chút thời gian)...")

# Định giá giải tích Heston (Fourier) làm mốc so sánh
df_clean['price_heston_analytical'] = df_clean.apply(
    lambda row: heston_analytical_price(
        row['underlying_price_S'], row['strike_K'], row['time_to_maturity_T'],
        row['risk_free_rate_r'], v0, kappa, theta, xi, rho, row['option_type']
    ), axis=1
)

# Định giá bằng Monte Carlo
df_clean['price_heston_monte_carlo'] = df_clean.apply(
    lambda row: heston_monte_carlo_price(
        row['underlying_price_S'], row['strike_K'], row['time_to_maturity_T'],
        row['risk_free_rate_r'], v0, kappa, theta, xi, rho, row['option_type'],
        N_steps=100, N_paths=20000
    ), axis=1
)

In [ ]:
# 4. Tính toán sai số để phục vụ đánh giá chương thực nghiệm
df_clean['mc_error_abs'] = np.abs(df_clean['price_heston_analytical'] - df_clean['price_heston_monte_carlo'])
df_clean['mc_error_rel'] = df_clean['mc_error_abs'] / np.maximum(df_clean['price_heston_analytical'], 1.0)

mae = df_clean['mc_error_abs'].mean()
mre = df_clean['mc_error_rel'].mean() * 100

print(f"✅ Mô phỏng hoàn tất!")
print(f" 🔹 Sai số tuyệt đối trung bình (MAE): {mae:.4f} USD")
print(f" 🔹 Sai số tương đối trung bình (MRE): {mre:.2f}%")

# 5. Lưu trữ kết quả
df_clean.to_csv("../data/processed/03_heston_monte_carlo_prices.csv", index=False)
print("💾 Kết quả định giá đã được lưu vào 'data/processed/03_heston_monte_carlo_prices.csv'")

In [ ]:
# 6. Vẽ đồ thị phân bổ sai số theo Moneyness (S/K)
df_clean['moneyness'] = df_clean['underlying_price_S'] / df_clean['strike_K']

fig, ax = plt.subplots(figsize=(11, 6))
call_mask = df_clean['option_type'] == 'call'
put_mask  = df_clean['option_type'] == 'put'

ax.scatter(df_clean.loc[call_mask, 'moneyness'],
           df_clean.loc[call_mask, 'mc_error_rel'] * 100,
           c='steelblue', alpha=0.6, s=20, label='Call')
ax.scatter(df_clean.loc[put_mask,  'moneyness'],
           df_clean.loc[put_mask,  'mc_error_rel'] * 100,
           c='tomato',    alpha=0.6, s=20, label='Put')

ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1.2, label='ATM (S/K = 1)')
ax.axhline(y=mre, color='orange', linestyle=':', linewidth=1.5,
           label=f'MRE trung bình ({mre:.2f}%)')

ax.set_title('Monte Carlo Relative Error vs. Moneyness (S/K)', fontsize=14, fontweight='bold')
ax.set_xlabel('Moneyness  S / K', fontsize=12)
ax.set_ylabel('Relative Error (%)', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/mc_error_by_moneyness.png', dpi=300, bbox_inches='tight')
print("📊 Đồ thị phân bổ sai số đã được lưu vào 'outputs/plots/mc_error_by_moneyness.png'")
plt.show()